In [28]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
from typing import List, Optional
import pandas as pd
import pyarrow as pya
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import glob
from itertools import chain
import numpy as np
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os, urllib.parse, urllib.request
import codecs

In [29]:

def hex_to_bytes(h):
    #Convert hex/bytes/string to bytes.
    if h is None or (isinstance(h, float) and np.isnan(h)): 
        return b""
    if isinstance(h, (bytes, bytearray)): 
        return bytes(h)
    s = str(h).strip().replace("0x","").replace(" ","")
    if s == "": 
        return b""
    if len(s) % 2 == 1: 
        s = "0"+s
    try: 
        return bytes.fromhex(s)
    except Exception: 
        print(f"Warning: invalid hex input: {h}")
        return str(h).encode("utf-8", errors="ignore")


def hamming_distance(a: bytes, b: bytes) -> int:
    
    
    #len_mismatch = (len(a) != len(b))

    # Pad to same length
    max_len = max(len(a), len(b))
    a_padded = a + b'\x00' * (max_len - len(a))
    b_padded = b + b'\x00' * (max_len - len(b))
    
    # Count differences
    #dist = sum(byte_a != byte_b for byte_a, byte_b in zip(a_padded, b_padded))
    
    distance = 0
    for byte_a, byte_b in zip(a_padded, b_padded):
        distance += bin(byte_a ^ byte_b).count('1')
    return (distance)
    
    #return (dist, len_mismatch)

In [30]:

def compute_hamming_distances_training(dumps, out_csv, process_per_dump=True):
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    
    records = []
    prev_payload = {}  # {arbitration_id: previous_bytes}
    
    for dump_name, df in dumps:
        d = df.copy()
        
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
            else:
                d = d.reset_index().rename(columns={"index": "timestamp"})
        
        
        d = d.sort_values("timestamp", kind="mergesort")
        has_label = "label" in d.columns
        
        for _, row in d.iterrows():
            aid = row["arbitration_id"]
            ts = row["timestamp"]
            lab = int(row["label"]) if has_label and not pd.isna(row["label"]) else 0
            curr_bytes = hex_to_bytes(row["data"])
            
            # Get previous payload for this ID
            prev = prev_payload.get(aid)
            
            if prev is not None:
                # Compute Hamming distance
                dist = hamming_distance(curr_bytes, prev)
                #max_len = max(len(curr_bytes), len(prev))
                #norm_dist = dist / (max_len * 8) if max_len > 0 else 0.0
                
                records.append({
                    "dump": dump_name,
                    "timestamp": ts,
                    "arbitration_id": aid,
                    "payload_len": len(curr_bytes),
                    "ham_dist": dist,
                    #"ham_norm": norm_dist,
                    #"len_mismatch": len_mismatch,  
                    "label": lab
                })
            
            

            # Update previous payload
            prev_payload[aid] = curr_bytes
        #False for the bening so we can compute the distances using all 7 dumps
        if process_per_dump:
            prev_payload.clear()
    
    out_df = pd.DataFrame.from_records(records)
    out_df.to_csv(out_csv, index=False)
    print(f" Saved: {out_csv} (rows={len(out_df):,})")
    
    return out_df

In [31]:
def compute_hamming_distances_testing(row, prev_payload: Dict, ranges_indexed):
    
    aid = row.arbitration_id
    lab = int(row.label) if hasattr(row, 'label') and pd.notna(row.label) else 0
    data = row.data
    curr_bytes = hex_to_bytes(data)
    
    # Get previous payload for this ID
    prev = prev_payload.get(aid)
    
    if prev is None:
        prev_payload[aid] = curr_bytes
        return None, lab, prev_payload
    
    
    # Compute Hamming distance
    dist = hamming_distance(curr_bytes, prev)
    #should_update = True
    #min_val = None  #INITIALIZE
    #max_val = None  
    
    
    """if ranges_indexed is not None and aid in ranges_indexed.index:
        min_val = ranges_indexed.at[aid, 'min_hamming']
        max_val = ranges_indexed.at[aid, 'max_hamming']
    
    if (min_val is not None) and (max_val is not None):
        if min_val <= dist <= max_val:
            should_update = True  #inside range
        else:
            should_update = False
            
    if should_update:"""
    prev_payload[aid] = curr_bytes
    #ASK PROFESSOR - what to do if the previous payload is malicious because you compare a valid one with a malicious that you saved in the payload_prev and you get a FP????
    return dist, lab, prev_payload
   

In [32]:

def build_hamming_range_model(hamming_csv, output_csv):
    
    
    df = pd.read_csv(hamming_csv)
    
    # Group by ID and compute min/max
    ranges = (df.groupby('arbitration_id')['ham_dist']
            .agg(['min', 'max', 'count'])
            .reset_index())
    
    ranges.columns = ['arbitration_id', 'min_hamming', 'max_hamming', 'n_samples']
    
    # Compute range size
    ranges['range_size'] = ranges['max_hamming'] - ranges['min_hamming']
    
    # Normalized versions (for 8-byte payloads, max=8)
    ranges['min_norm'] = ranges['min_hamming'] / 64.0
    ranges['max_norm'] = ranges['max_hamming'] / 64.0
    ranges['range_norm'] = ranges['range_size'] / 64.0
    
    # Classify IDs (paper uses sigma=6 for byte-level Hamming)
    sigma = 6 * 4
    """def classify(r):
        if r == 0:
            return 'NoRange'
        elif r <= sigma:
            return 'SmallRange'
        else:
            return 'MidRange'
    
    ranges['class'] = ranges['range_size'].apply(classify)
    
    # Expected detection rates 
    def expected_tpr(cls):
        if cls == 'NoRange':
            return 0.98  
        elif cls == 'SmallRange':
            return 0.97  
        else:
            return 0.25  
    
    ranges['expected_tpr'] = ranges['class'].apply(expected_tpr)
    
    # Sort by range size
    ranges = ranges.sort_values('range_size')"""
    
    # Save
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    ranges.to_csv(output_csv, index=False)
    
    # Print summary
    print(f"\nLearned ranges for {len(ranges)} unique IDs")
    print(f"   Saved to: {output_csv}")

    
    print("="*70)
    return ranges


In [33]:

def detect_simple(dist, arb_id, label, ranges_lookup):
    """
    Simple detection: Flag if distance is OUTSIDE [min, max] range.
    No windowing - direct per-message detection.
    
    Args:
        test_csv: Path to test data with Hamming distances
        ranges_csv: Path to learned ranges
        output_csv: Where to save results
        
    Returns:
        DataFrame with detection results and performance metrics
    """
    #check_anomaly_and_labels = []
    # Load data
    if arb_id not in ranges_lookup:
        return {
            'arbitration_id': arb_id,
            'hamming_distance': dist,
            'detected': False,  # Unknown ID = anomaly
            'label': label,
            'reason': 'unknown_id'
        }
    
    min_ham, max_ham = ranges_lookup[arb_id]
    detected = (dist < min_ham) or (dist > max_ham)
    
    return {
        'arbitration_id': arb_id,
        'hamming_distance': dist,
        'detected': detected,
        'label': label,
        'min_range': min_ham,
        'max_range': max_ham
    }

    
    # Core detection logic from paper:
    # Anomaly if distance is OUTSIDE [min, max]
    detected = (dist < min_ham) or (dist > max_ham)
    
    
    return {
        'arbitration_id': arb_id,
        'hamming_distance': dist,
        'detected': detected,
        'label': label,
        'min_range': min_ham,
        'max_range': max_ham
    }
    """#-----------------
    #-----------------
    #-----------------
    # Check where your FPs are coming from:
    fps = test_with_ranges[
        (test_with_ranges['label'] == 0) & 
        (test_with_ranges['is_anomaly'] == True)
    ]

    print(f"False Positives: {len(fps)}")
    print(f"\nFP Breakdown:")
    print(f"  Length mismatches: {fps['len_mismatch'].sum()}")
    print(f"  Hamming too low: {(fps['ham_dist'] < fps['min_hamming']).sum()}")
    print(f"  Hamming too high: {(fps['ham_dist'] > fps['max_hamming']).sum()}")

    # Check which IDs produce FPs:
    fp_ids = fps.groupby('arbitration_id').size().sort_values(ascending=False)
    print(f"\nTop FP-producing IDs:")
    print(fp_ids.head(10))"""
    
    #-----------------
    #-----------------
    #-----------------
    """
    # Count anomalies
    n_anomalies = test_with_ranges['is_anomaly'].sum()
    n_total = len(test_with_ranges)
    
    print(f"\n Detection Summary:")
    print(f"   Total messages:    {n_total:,}")
    print(f"   Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.2f}%)")
    
    # Compute metrics if labels available
    metrics = {}
    if 'label' in test_with_ranges.columns:
        y_true = test_with_ranges['label'].values
        y_pred = test_with_ranges['is_anomaly'].values
        
        TP = ((y_true == 1) & (y_pred == True)).sum()
        FP = ((y_true == 0) & (y_pred == True)).sum()
        TN = ((y_true == 0) & (y_pred == False)).sum()
        FN = ((y_true == 1) & (y_pred == False)).sum()
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        print(f"\n Detection Results:")
        print(f"   TPR: {tpr*100:.2f}% | FPR: {fpr*100:.2f}% | F1: {f1*100:.2f}%")
        print(f"   TP: {TP:,} | FP: {FP:,} | TN: {TN:,} | FN: {FN:,}")
        
        metrics['overall'] = {
            'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
            'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
        }
    
    # Save results
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    test_with_ranges.to_csv(output_csv, index=False)
    
    return test_with_ranges, metrics"""


In [34]:
def compute_metrics_from_detections(detections_list):
    
    #Compute metrics from list of detection results.
    
    if not detections_list:
        return pd.DataFrame(), {}
    
    detections_df = pd.DataFrame(detections_list)
    
    # Count anomalies
    n_anomalies = detections_df['detected'].sum()
    n_total = len(detections_df)
    
    print(f"\n Detection Summary:")
    print(f"   Total messages:    {n_total:,}")
    print(f"   Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.2f}%)")
    
    # Compute metrics if labels available
    metrics = {}
    if 'label' in detections_df.columns and detections_df['label'].notna().any():
        y_true = detections_df['label'].values
        y_pred = detections_df['detected'].values
        
        TP = ((y_true == 1) & (y_pred == True)).sum()
        FP = ((y_true == 0) & (y_pred == True)).sum()
        TN = ((y_true == 0) & (y_pred == False)).sum()
        FN = ((y_true == 1) & (y_pred == False)).sum()
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        print(f"\n Detection Results:")
        print(f"   TPR: {tpr*100:.2f}% | FPR: {fpr*100:.2f}% | F1: {f1*100:.2f}%")
        print(f"   TP: {TP:,} | FP: {FP:,} | TN: {TN:,} | FN: {FN:,}")
        
        metrics['overall'] = {
            'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
            'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
        }
    else:
        print("\nNo labels available - cannot compute metrics")
    
    return detections_df, metrics


In [35]:

if __name__ == "__main__":

    # take all masw files
    files_pathname = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\thresholds_test"
    fabr_files = [
    f for f in os.listdir(files_pathname)
    if f.startswith("dump6-fabr-") and f.endswith(".parquet")
]
    dump6 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump6.parquet').to_pandas()

    fabr_ids = [f.split('-')[-1].removesuffix('.parquet') for f in fabr_files]
    #masq_ids = ["112h","113h","200h"]
    print(f"Found {len(fabr_ids)} fabrication files.")
    print("Fabrication IDs in discovery order:", fabr_ids)
    
    #build ranges
    ranges = build_hamming_range_model(
    "artifacts/benign_hamming.csv",
    "artifacts2/hamming_ranges_fabr_stream.csv"
)
    print("ranges created")
    ranges_df = pd.read_csv("artifacts2/hamming_ranges_fabr_stream.csv")
    # ========================================================================
    ranges_indexed = ranges.set_index('arbitration_id')
    results_summary = []
    #i made this dict to speed up the lookup of ranges instead of querying the dataframe each time cause it took 4h per file 
    ranges_lookup = {
    row['arbitration_id']: (row['min_hamming'], row['max_hamming'])
    for _, row in ranges_df.iterrows()
}
    
    for i, fabr_id in enumerate(fabr_ids, 1):  
        print(f"\n[{i}/{len(fabr_ids)}] Testing: dump6-fabr-{fabr_id}")  
        #just to know at which row i am rn
        #j = 0
        try:
            # Load attack file
            attack_file = os.path.join(files_pathname, f"dump6-fabr-{fabr_id}.parquet") 
            attack_df = pq.read_table(attack_file).to_pandas()
            
            print(f"   Loaded: {len(attack_df):,} messages")
            prev_payload = {}
            attack_dump = attack_df.copy()
            
            #populating the prev_payload with the benign data at the start _ASK PROFESSOR IF WISE TO DO THAT
            print("   Seeding baseline with benign traffic...")
            dump6_sorted = dump6.sort_values("timestamp")
            for row in dump6_sorted.itertuples():
                aid = row.arbitration_id
                curr_bytes = hex_to_bytes(row.data)
                prev_payload[aid] = curr_bytes

            print(f"   Baseline seeded: {len(prev_payload)} IDs")
            
            #-----------------------------------
            
            
            if "timestamp" not in attack_dump.columns:
                if attack_dump.index.name == "timestamp":
                   attack_dump = attack_dump.reset_index()
                else:
                    attack_dump = attack_dump.reset_index().rename(columns={"index": "timestamp"})
                    
            attack_dump = attack_dump.sort_values("timestamp", kind="mergesort")
            
            detect_list = []
            #prev_payload= {}
            # Compute Hamming
            for row in attack_dump.itertuples():
                
                #debugging progress
                """if idx % 10000 == 0:
                    print(f"\r   Processing packets... {idx:,}/{len(attack_df):,}", end="", flush=True)"""
                
                #now here it should return just the distance and the label
                current_aid = row.arbitration_id
                attack_ham_dist, lab, prev_payload = compute_hamming_distances_testing(
                    row, 
                    prev_payload,
                    ranges_indexed # Pass current state
                )
            
                # Detect
                if attack_ham_dist is not None:
                    results = detect_simple(
                        attack_ham_dist,
                        current_aid,
                        lab,  
                        ranges_lookup
                    )
                    detect_list.append(results)
                    
                 # Compute metrics for this attack file
            detections_df, metrics = compute_metrics_from_detections(detect_list)
            
            # Save detections
            output_csv = f"artifacts2/detections_fabr_{fabr_id}.csv"
            Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
            detections_df.to_csv(output_csv, index=False)
            # Store results
            if 'overall' in metrics:
                results_summary.append({
                    'fabr_id': fabr_id,  
                    'n_messages': len(attack_df),
                    'n_attacks': metrics['overall']['TP'] + metrics['overall']['FN'],
                    'TPR': metrics['overall']['TPR'] * 100,
                    'FPR': metrics['overall']['FPR'] * 100,
                    'Precision': metrics['overall']['Precision'] * 100,
                    'F1': metrics['overall']['F1'] * 100,
                    'TP': metrics['overall']['TP'],
                    'FP': metrics['overall']['FP'],
                    'TN': metrics['overall']['TN'],
                    'FN': metrics['overall']['FN']
                })
                
                """print(f"   TPR: {metrics['overall']['TPR']*100:.1f}%, "
                    f"FPR: {metrics['overall']['FPR']*100:.2f}%, "
                    f"F1: {metrics['overall']['F1']*100:.1f}%")"""
        
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # ========================================================================
    # STEP 4: Summary
    # ========================================================================
    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv("artifacts2/all_masq_summary_stream.csv", index=False)
    
    print("\n" + "="*70)
    print("Fabrication ATTACKS - FINAL SUMMARY")
    print("="*70)
    print(f"Tested: {len(summary_df)}/{len(fabr_ids)} attack types")
    
    if len(summary_df) > 0:
        print(f"\nOverall Performance:")
        print(f"   Mean TPR:       {summary_df['TPR'].mean():.2f}%")
        print(f"   Median TPR:     {summary_df['TPR'].median():.2f}%")
        print(f"   Min TPR:        {summary_df['TPR'].min():.2f}%")
        print(f"   Max TPR:        {summary_df['TPR'].max():.2f}%")
        print(f"   Mean FPR:       {summary_df['FPR'].mean():.4f}%")
        print(f"   Mean F1:        {summary_df['F1'].mean():.2f}%")
        
        print("\nResults by Masq Id:")
        print(summary_df[['fabr_id', 'n_attacks', 'TPR', 'FPR', 'F1']].to_string(index=False))

Found 35 fabrication files.
Fabrication IDs in discovery order: ['044h', '080h', '081h', '111h', '112h', '113h', '162h', '18Fh', '200h', '220h', '251h', '260h', '2B0h', '316h', '329h', '381h', '383h', '386h', '387h', '47Fh', '4F1h', '50Ch', '52Ah', '541h', '545h', '547h', '549h', '553h', '555h', '556h', '557h', '58Bh', '593h', '5A0h', '5B0h']

Learned ranges for 64 unique IDs
   Saved to: artifacts2/hamming_ranges_fabr_stream.csv
ranges created

[1/35] Testing: dump6-fabr-044h
   Loaded: 4,280,937 messages
   Seeding baseline with benign traffic...
   Baseline seeded: 64 IDs

 Detection Summary:
   Total messages:    4,280,937
   Flagged anomalies: 1,984 (0.05%)

 Detection Results:
   TPR: 100.00% | FPR: 0.02% | F1: 68.26%
   TP: 1,028 | FP: 956 | TN: 4,278,953 | FN: 0

[2/35] Testing: dump6-fabr-080h
   Loaded: 4,375,918 messages
   Seeding baseline with benign traffic...
   Baseline seeded: 64 IDs

 Detection Summary:
   Total messages:    4,375,918
   Flagged anomalies: 95,294 (2.1

In [36]:
# For good detection (100% TPR)
fabr_good = pd.read_csv("artifacts2/detections_fabr_044h.csv")
attacks_good = fabr_good[fabr_good['label'] == 1]

print("ID 044h (100% TPR):")
print(f"  Attack distances: {attacks_good['hamming_distance'].describe()}")
print(f"  Unique distances: {attacks_good['hamming_distance'].unique()}")

# For poor detection (10% TPR)
fabr_bad = pd.read_csv("artifacts2/detections_fabr_162h.csv")
attacks_bad = fabr_bad[fabr_bad['label'] == 1]

print("\nID 162h (10% TPR):")
print(f"  Attack distances: {attacks_bad['hamming_distance'].describe()}")
print(f"  Unique distances: {attacks_bad['hamming_distance'].unique()}")

# Load actual payloads
fabr_raw_good = pq.read_table("X-CANIDS/raw/dump6-fabr-044h.parquet").to_pandas()
fabr_raw_bad = pq.read_table("X-CANIDS/raw/dump6-fabr-162h.parquet").to_pandas()

print("\n044h Attack Payloads:")
print(fabr_raw_good[fabr_raw_good['label']==1]['data'].unique())

print("\n162h Attack Payloads:")
print(fabr_raw_bad[fabr_raw_bad['label']==1]['data'].unique())

ID 044h (100% TPR):
  Attack distances: count    1028.000000
mean       34.238327
std        13.434924
min        20.000000
25%        21.000000
50%        42.000000
75%        43.000000
max        64.000000
Name: hamming_distance, dtype: float64
  Unique distances: [21 43 20 64 44 42 22 41 23]

ID 162h (10% TPR):
  Attack distances: count    95998.000000
mean        34.353872
std         23.291781
min          2.000000
25%         10.000000
50%         51.000000
75%         56.000000
max         64.000000
Name: hamming_distance, dtype: float64
  Unique distances: [12 51 13 14 53 11 64 50 52 54 10 15 49  6 58  8 57  7 55 56  9 61  5 59
 47 16 48  4 60  3 62  2 17 46 18]


FileNotFoundError: X-CANIDS/raw/dump6-fabr-044h.parquet

In [ ]:
# Check the learned ranges
ranges_df = pd.read_csv("artifacts2/hamming_ranges_fabr_stream.csv")

range_044h = ranges_df[ranges_df['arbitration_id'] == 0x044]
range_162h = ranges_df[ranges_df['arbitration_id'] == 0x162]

print("ID 044h range:")
print(range_044h[['min_hamming', 'max_hamming', 'range_size']])

print("\nID 162h range:")
print(range_162h[['min_hamming', 'max_hamming', 'range_size']])

ID 044h range:
   min_hamming  max_hamming  range_size
2            0            8           8

ID 162h range:
    min_hamming  max_hamming  range_size
10            0           18          18
